In [1]:
import os, requests, hashlib
from pathlib import Path
from tqdm import tqdm

os.system("df -h /kaggle/working")

WORKDIR = Path("/kaggle/working/boucheron_download")
WORKDIR.mkdir(parents=True, exist_ok=True)

FILES = [
    ("https://zenodo.org/records/7807466/files/C1.0_24hr_224_png_Labels.txt?download=1",
     "C1.0_24hr_224_png_Labels.txt", "c07787bd9def72d0519a49c02cb86f7c"),
    ("https://zenodo.org/records/7807466/files/Lat60_Lon60_Nans0_C1.0_24hr_png_224_features.csv?download=1",
     "Lat60_Lon60_Nans0_C1.0_24hr_png_224_features.csv", "95963729b452c90e3d0c3c1ec8822ba4"),
    ("https://zenodo.org/records/7807466/files/List_of_AR_in_Test_Data_by_AR.csv?download=1",
     "List_of_AR_in_Test_Data_by_AR.csv", "6f73d64b63429ee26876a156a734fa12"),
    ("https://zenodo.org/records/7807466/files/List_of_AR_in_Train_Data_by_AR.csv?download=1",
     "List_of_AR_in_Train_Data_by_AR.csv", "0b8b0b42eba839007bd40f163fc94455"),
    ("https://zenodo.org/records/7807466/files/List_of_AR_in_Validation_data_by_AR.csv?download=1",
     "List_of_AR_in_Validation_data_by_AR.csv", "fa192a397d0b582d372270af9db26393"),
    ("https://zenodo.org/records/7807466/files/README.md?download=1",
     "README.md", "ec8fa56a306a2e70862332dd94efb849"),
    ("https://zenodo.org/records/7807466/files/Test_Data_by_AR_png_224.csv?download=1",
     "Test_Data_by_AR_png_224.csv", "d6f0abaca77628def3462366e0937639"),
    ("https://zenodo.org/records/7807466/files/Train_Data_by_AR_png_224.csv?download=1",
     "Train_Data_by_AR_png_224.csv", "58442b71ebc3741a19cba18c75061017"),
    ("https://zenodo.org/records/7807466/files/Validation_Data_by_AR_png_224.csv?download=1",
     "Validation_Data_by_AR_png_224.csv", "2f4ec494748861ed110edf23b8bdf636"),
]

IMAGE_ARCHIVE = (
    "https://zenodo.org/records/7775776/files/Lat60_Lon60_Nans0_png_224.tar.gz?download=1",
    "Lat60_Lon60_Nans0_png_224.tar.gz",
    "c6498435e19705710ec4e1ec0c041a1d",
)

def md5sum(path, chunk=1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def download_and_verify(url, filename, expected_md5, dest_dir=WORKDIR):
    dest = dest_dir / filename
    with requests.get(url, stream=True, allow_redirects=True, timeout=60) as r:
        r.raise_for_status()
        total = int(r.headers.get("Content-Length", 0))
        with open(dest, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=filename) as pbar:
            for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                f.write(chunk)
                pbar.update(len(chunk))
    actual_md5 = md5sum(dest)
    ok = actual_md5 == expected_md5
    print(f"{filename}: {'MD5 OK' if ok else 'MD5 MISMATCH!'}")
    if not ok:
        raise RuntimeError(f"Checksum failed for {filename}")
    return dest

# small files first
for url, fname, md5 in FILES:
    download_and_verify(url, fname, md5)

# the big 13GB image file
download_and_verify(*IMAGE_ARCHIVE)

print("ALL FILES DOWNLOADED AND VERIFIED")
os.system(f"du -sh {WORKDIR}")
os.system("df -h /kaggle/working")

Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   68K   20G   1% /kaggle/working


C1.0_24hr_224_png_Labels.txt: 57.6MB [00:03, 19.1MB/s]


C1.0_24hr_224_png_Labels.txt: MD5 OK


Lat60_Lon60_Nans0_C1.0_24hr_png_224_features.csv: 465MB [00:52, 8.90MB/s] 


Lat60_Lon60_Nans0_C1.0_24hr_png_224_features.csv: MD5 OK


...

Validation_Data_by_AR_png_224.csv: 6.24MB [00:00, 11.6MB/s]


Validation_Data_by_AR_png_224.csv: MD5 OK


Lat60_Lon60_Nans0_png_224.tar.gz: 100%|██████████| 13.0G/13.0G [39:26<00:00, 5.48MB/s]  


Lat60_Lon60_Nans0_png_224.tar.gz: MD5 OK
ALL FILES DOWNLOADED AND VERIFIED
13G	/kaggle/working/boucheron_download
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   13G  6.9G  65% /kaggle/working


0

In [2]:
metadata = '''{
  "title": "Boucheron Solar Flare Magnetograms C1.0 24hr 224px",
  "id": "esayasmelaku/solar-flare-boucheron-c1-224",
  "licenses": [{"name": "CC-BY-SA-4.0"}]
}'''

with open("/kaggle/working/boucheron_download/dataset-metadata.json", "w") as f:
    f.write(metadata)

print("metadata file created")

metadata file created


In [3]:
import os
os.system("kaggle datasets create -p /kaggle/working/boucheron_download --dir-mode zip")

Starting upload for file List_of_AR_in_Validation_data_by_AR.csv


100%|██████████| 785/785 [00:00<00:00, 3.70kB/s]
  4%|▍         | 17.2M/444M [00:00<00:02, 180MB/s]

Upload successful: List_of_AR_in_Validation_data_by_AR.csv (785B)
Starting upload for file Lat60_Lon60_Nans0_C1.0_24hr_png_224_features.csv


100%|██████████| 444M/444M [00:04<00:00, 95.7MB/s] 
  0%|          | 0.00/5.87M [00:00<?, ?B/s]

Upload successful: Lat60_Lon60_Nans0_C1.0_24hr_png_224_features.csv (444MB)
Starting upload for file Test_Data_by_AR_png_224.csv


...

100%|██████████| 6.13k/6.13k [00:00<00:00, 27.4kB/s]
  0%|          | 0.00/10.1k [00:00<?, ?B/s]

Upload successful: List_of_AR_in_Train_Data_by_AR.csv (6KB)
Starting upload for file README.md


100%|██████████| 10.1k/10.1k [00:00<00:00, 36.1kB/s]


Upload successful: README.md (10KB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/esayasmelaku/solar-flare-boucheron-c1-224


0

In [18]:
import pandas as pd
from pathlib import Path

DATA_ROOT = Path("/kaggle/input/datasets/esayasmelaku/solar-flare-boucheron-c1-224")

# List everything
for item in DATA_ROOT.iterdir():
    print(item)

# Read the labels/features CSV
labels_csv = DATA_ROOT / "Lat60_Lon60_Nans0_C1.0_24hr_png_224_features.csv"
df = pd.read_csv(labels_csv, header=None)
print(df.shape)
print(df.head())

df.columns = [f"feature_{i}" for i in range(29)] + ["label", "flare_size", "filename"]
print(df["label"].value_counts())

/kaggle/input/datasets/esayasmelaku/solar-flare-boucheron-c1-224/List_of_AR_in_Train_Data_by_AR.csv
/kaggle/input/datasets/esayasmelaku/solar-flare-boucheron-c1-224/List_of_AR_in_Validation_data_by_AR.csv
/kaggle/input/datasets/esayasmelaku/solar-flare-boucheron-c1-224/README.md
/kaggle/input/datasets/esayasmelaku/solar-flare-boucheron-c1-224/List_of_AR_in_Test_Data_by_AR.csv
/kaggle/input/datasets/esayasmelaku/solar-flare-boucheron-c1-224/Lat60_Lon60_Nans0_C1.0_24hr_png_224_features.csv
/kaggle/input/datasets/esayasmelaku/solar-flare-boucheron-c1-224/Validation_Data_by_AR_png_224.csv
/kaggle/input/datasets/esayasmelaku/solar-flare-boucheron-c1-224/Lat60_Lon60_Nans0_png_224
/kaggle/input/datasets/esayasmelaku/solar-flare-boucheron-c1-224/Train_Data_by_AR_png_224.csv
/kaggle/input/datasets/esayasmelaku/solar-flare-boucheron-c1-224/Test_Data_by_AR_png_224.csv
/kaggle/input/datasets/esayasmelaku/solar-flare-boucheron-c1-224/C1.0_24hr_224_png_Labels.txt
(950047, 32)
         0         1   

In [20]:
# Use .iloc to check by POSITION, regardless of current column names
for pos in [28, 29, 30]:
    print(f"Column at position {pos} (name: {df.columns[pos]}) value_counts (top 5):")
    print(df.iloc[:, pos].value_counts().head())
    print()

Column at position 28 (name: feature_28) value_counts (top 5):
feature_28
57734.0    33
57035.0    33
46787.0    33
53206.0    33
49586.0    33
Name: count, dtype: int64

Column at position 29 (name: label) value_counts (top 5):
label
0    759465
1    190582
Name: count, dtype: int64

Column at position 30 (name: flare_size) value_counts (top 5):
flare_size
0       759465
C1.1      9644
C1.2      8575
C1.3      7990
C1.0      7976
Name: count, dtype: int64



In [21]:
df.columns = [f"feature_{i}" for i in range(29)] + ["label", "flare_size", "filename"]
print(df.columns.tolist())
print(df[["label", "flare_size", "filename"]].head())

['feature_0', 'feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9', 'feature_10', 'feature_11', 'feature_12', 'feature_13', 'feature_14', 'feature_15', 'feature_16', 'feature_17', 'feature_18', 'feature_19', 'feature_20', 'feature_21', 'feature_22', 'feature_23', 'feature_24', 'feature_25', 'feature_26', 'feature_27', 'feature_28', 'label', 'flare_size', 'filename']
   label flare_size                                           filename
0      0          0  1064_hmi.M_720s.20100501_000000_TAI.1.magnetog...
1      0          0  1064_hmi.M_720s.20100501_001200_TAI.1.magnetog...
2      0          0  1064_hmi.M_720s.20100501_002400_TAI.1.magnetog...
3      0          0  1064_hmi.M_720s.20100501_003600_TAI.1.magnetog...
4      0          0  1064_hmi.M_720s.20100501_004800_TAI.1.magnetog...
